# Exploring Image and Text-to-Image Embeddings

<a target="_blank" href="https://colab.research.google.com/github/impresso/impresso-datalab-notebooks/blob/main/workshop_resources/ws4-embeddings/explore_multimodal-embeddings.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

If something doesn't work, you can [report a problem](https://github.com/impresso/impresso-datalab-notebooks/blob/main/reporting-problems.md).

## What is this notebook about?

This notebook demonstrates how to explore historical image collections using Impresso’s text-to-image and image-only embeddings, from keyword search to visual similarity retrieval.

In the **first section**, we begin with Open-CLIP to perform text-to-image search. We start by choosing a small set of keywords and using them to retrieve relevant images. Next, we expand the keywords into longer textual descriptions and repeat the process, allowing us to observe how richer language produces more nuanced and precise results.

In the **second section**, we work with DinoV2 image-only embeddings to identify visual similarities within the collection. Given a single reference image, we search for visually related items and interpret what features the model captures.

We will explore **how radio is represented both in images, and in the programs**. This will allow us to explore the image and textual elements using both types of embeddings.

## What you will learn?

- Perform keyword-based image retrieval using image captions and Open-CLIP, and convert a text query into an embedding for text-to-image similarity search;
- Use DINOv2 to search for visual similarities directly from a reference image;
- Compare how multimodal embeddings and visual-only embeddings support different research strategies.

## Useful resources

- [Impresso Python Library](https://impresso.github.io/impresso-py/)
- [Impresso Hugging Face](https://ipyleaflet.readthedocs.io/en/latest/index.html)

## Prerequisites

Run the following cells to install the required package and to connect to Imrpesso API:

> If you are working with Google Colab, you may need to restart the kernel. Go to *Runtime* and select *Restart session*.

In [2]:
# Impresso Python package with embeddings search feature
!pip install impresso

In [ ]:

# Connecting to Impresso API
from impresso import connect, OR, AND, DateRange
client = connect()

> In this notebook, we will often move back and forth between the notebook and the Impresso App, so a small function for constructing links is useful. Due to copyright restrictions, images might not be fully displayed here but you can access them via the Impresso Web App.

In [ ]:
# Function to generate webapp URLs for images

def img_webapp_url(uid, issue_mode=True):
  mode = "issue" if issue_mode else "search/images"
  pre, suf = uid.split('-a-')
  suffix = f"{pre}-a/view?articleId={suf}" if issue_mode else uid
  return f'https://dev.impresso-project.ch/app/{mode}/{suffix}'

# Text-to-Image embeddings with Open-Clip

First, we examine how the system retrieves images from simple keywords or short phrases, providing an initial sense of the most similar results.

## 1. Keyword search on image captions

In [3]:
kw_radio = 'radio'

result = client.images.find(term=kw_radio)
result

## 2. Text-to-image similarity search with Open-Clip

Next, we embed the same keyword with Open-CLIP and use this embedding to search through the Open-CLIP image embeddings, enabling text-to-image similarity retrieval.

In [4]:
kw_embedding = client.tools.embed_text(text=kw_radio, target='multimodal')
kw_embedding

> Having inspected the generated embedding, one might wonder what these weird characters and numbers mean: ```openclip-768:1zuRvAvlzLvJJxw9tRKdO5yyaryrbIK9wuLOvDO12bz...```
The reason for why this embedding does not look like a vector of numbers is rather simple: **It's encoded in a data-efficient format**.

In [5]:
# Searching images similar to the keywordembeddings
results = client.images.find(
  embedding=kw_embedding,
  limit=6
)
results

In [ ]:
results.df[['issueId', 'imageTypes.visualContentType']]

In [ ]:
import numpy as np
import pandas as pd

# Print the URLs for the first 5 images
for uid, r in results.df.head(5).iterrows():
  print(f"Result {uid} - link to image CI {r.previewUrl} - type {r['imageTypes.visualContentType']}")
  if pd.notna(r.contentItemId):
    print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

> The extracted images either feature iillustrations of radios (physical radio sets) or illustrated headers of radio sections.
We can try to filter by image type, such as `Object`, `Non-Figurative Visual Content` and `Ornament or Illustrated Title`.


In [8]:
# Filter results based on the image content type "Object"
object_results = client.images.find(
  content_type="Object",
  embedding=kw_embedding,
  limit=5
)

# Print the URLs for the first 5 images
print(f"Results for images of type Object")
for uid, r in object_results.df.head(5).iterrows():
  print(f"Result {uid} - link to image CI {img_webapp_url(uid, issue_mode=False)}")
  if pd.notna(r.contentItemId):
    print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

> If you have opened all the links, you will notice that most of the elements from the previous search appear again, with the exception of the image `EXP-1960-03-31-a-i0096`. In addition, the images `EXP-2009-01-06-a-i0096` and `IMP-2009-01-06-a-i0080` were reused a few weeks later by the editors in `IMP-2010-02-02-a-i0120`.

In [9]:
# Filter results based on the image content type "Non-Figurative Visual Content" OR "Ornament or Illustrated Title"
non_fig_results = client.images.find(
  content_type=OR("Non-Figurative Visual Content", "Ornament or Illustrated Title"),
  embedding=kw_embedding,
  limit=5
)

# Combine results from both searches
combined_df = pd.concat([non_fig_results_1.df, non_fig_results_2.df])

# Drop the 'pageNumbers' column as it contains unhashable list objects, causing the TypeError
if 'pageNumbers' in combined_df.columns:
    combined_df = combined_df.drop(columns=['pageNumbers'])

non_fig_results_df = combined_df.drop_duplicates().head(5)

# Print the URLs for the first 5 images
print(f"Results for images of type Non-Figurative Visual Content or Ornament or Illustrated Title")
for uid, r in non_fig_results_df.iterrows():
  print(f"Result {uid} - link to image CI {img_webapp_url(uid, issue_mode=False)}")
  if pd.notna(r.contentItemId):
    print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

> There are far fewer results of this type, as they are generally rarer in the data. However, in both cases the model identifies the Radio section logo, likely because it also contains text.

## 3. Complex search queries with embeddings

Next, we refine our search by embedding a more **descriptive query** that targets the radio program section of a newspaper.

In [11]:
program_query = "Weekly radio program"

# Embed program_query prompt using Open-Clip model
pgm_embedding = client.tools.embed_text(text=program_query, target='multimodal')
pgm_embedding


# Print the URLs for images similar to the query embeddings
pgm_results = client.images.find(
  embedding=pgm_embedding,
  limit=6
)

for uid, r in pgm_results.df.head(5).iterrows():
  print(f"Result {uid} - link to image CI {img_webapp_url(uid, issue_mode=False)} - type {r['imageTypes.visualContentType']}")
  if pd.notna(r.contentItemId):
    print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

> We have successfully retrieved the illustrated section titles!
This query captures many of the radio program pages from L’Impartial in the late 1930s and early 1940s.

> Since **CLIP is multilingual**, we can try the same search using a query in French.

In [13]:
program_query_fr = "Programme Radio de la semaine"

# Embed program_query_fr prompt using Open-Clip model
pgm_fr_embedding = client.tools.embed_text(text=program_query_fr, target='multimodal')
pgm_fr_embedding


# Print the URLs for images similar to the query embeddings
pgm_fr_results = client.images.find(
  embedding=pgm_fr_embedding,
  limit=6
)

for uid, r in pgm_fr_results.df.head(5).iterrows():
  print(f"Result {uid} - link to image CI {img_webapp_url(uid, issue_mode=False)} - type {r['imageTypes.visualContentType']}")
  if str(r.contentItemId)!='nan':
    print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

> We obtain very similar results: most of them are program pages, but this time they are more recent and often list TV programs (note that the Swiss national radio and TV share the same name).
Now let’s see if we can go further and **retrieve actual images of radio stations**, ideally with people listening to the radio.


In [14]:
radio_query_fr = "Personnes écoutant la radio à côté du poste de radio."

# Embed radio_query_fr prompt using Open-Clip model
radio_fr_embedding = client.tools.embed_text(text=radio_query_fr, target='multimodal')
radio_fr_embedding


# Print the URLs for images similar to the query embeddings
radio_fr_results = client.images.find(
  embedding=radio_fr_embedding,
  limit=6
)

for uid, r in radio_fr_results.df.head(5).iterrows():

  print(f"Result {uid} - link to image CI {r.previewUrl} - type {r['imageTypes.visualContentType']}")
  if 'issueId' in r and str(r.contentItemId)!='nan':
    print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

> We retrieve more images of actual radio stations, often with people present. It can be useful to compare this with a similar sentence in English, or to refine the query to explicitly require a human figure in the scene.

In [15]:
radio_query_en = "People listening to a radio monitor."

# Embed radio_query_en prompt using Open-Clip model
radio_en_embedding = client.tools.embed_text(text=radio_query_en, target='multimodal')
radio_en_embedding


# Print the URLs for images similar to the query embeddings
radio_en_results = client.images.find(
  embedding=radio_en_embedding,
  limit=6
)

for uid, r in radio_en_results.df.head(5).iterrows():

  print(f"Result {uid} - link to image CI {r.previewUrl} - type {r['imageTypes.visualContentType']}")
  if 'issueId' in r and str(r.contentItemId)!='nan':
    print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

> Having a query in english seems to have done the trick here; as you can see most images are of the type human representation!
**Don't hesitate to explore further with different queries, more complex and simple ones, varying languages and using the help of the image type filter** to specify more precisely what's of interest!

# Image-only embeddings with DinoV2
Let's dive more into **image-to-image embeddings**, and search for images that match ones that are of particular interest to us.

## 1. Searching for similar images with an external image
Suppose we are interested in studying the **spread of new technologies in the 1980s**: in that case, the image `EXP-1983-08-31-a-i0208` from content item `EXP-1983-08-31-a-i0195` could serve as a useful reference to discover similar articles.


In [16]:
example_image_id = 'EXP-1983-08-31-a-i0208'

example_embedding = client.images.get_embeddings(example_image_id)
example_embedding[1]

In [17]:
dino_results = client.images.find(
  embedding=example_embedding[1],
  limit=7
)

# Print the URLs for images similar to the query embeddings
for uid, r in dino_results.df.head(6).iterrows():
  if uid != example_image_id:
    print(f"Result {uid} - link to image CI {r.previewUrl} - type {r['imageTypes.visualContentType']}")
    if 'contentItemId' in r and str(r.contentItemId)!='nan':
      print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

> We can see that the retrieved images are different, yet they share several key characteristics with our base image: they depict what appear to be mid to late-twentieth-century technologies and show people interacting with them.
However, the publication dates of the corresponding articles vary widely, ranging from the late 1950s to the mid-1990s. To narrow the results, we can apply a date filter to restrict the search to images published in issues from the mid-1970s to the mid-1990s. We can further refine the query by using the type filter introduced earlier to ensure that people are present in the images.

## 2. Searching for similar images with complex filters

In [18]:
# Filter results based on the image content type "Human Representation - Scene" OR "Human Representation - Portrait" AND date range 1975-1995
filter_results = client.images.find(
  content_type=OR("Human Representation - Scene", "Human Representation - Portrait"),
  embedding=example_embedding[1],
  date_range=DateRange("1975-01-01", "1995-01-01"),
  limit=7
)

# Print the URLs for images similar to the query embeddings
for uid, r in filter_results.df.head(6).iterrows():
  if uid != example_image_id:
    print(f"Result {uid} - link to image CI {r.previewUrl} - type {r['imageTypes.visualContentType']}")
    if 'contentItemId' in r and str(r.contentItemId)!='nan':
      print(f"       {r.contentItemId} - link to corresponding CI {img_webapp_url(r.contentItemId)}")

## 2. Searching for similar images with an external URL

Finally, if we find an image online that fits our research interests, **we can also use it directly as input for our search**. We simply need to embed the image using the same model — in this case, DINOv2, and then run the same similarity search as before.

For example, from the Wikimedia Commons category “People listening to radios”, we selected an [image](https://commons.wikimedia.org/wiki/Category:People_listening_to_radios) of a girl listening to the radio. To use it, we only need to choose a version of the [image](https://commons.wikimedia.org/wiki/File:REA,_%22Little_girl_by_radio%22_-_NARA_-_195876.tif) at an appropriate resolution - for instance, the 527 × 628 pixel version available on the image’s Wikimedia Commons page, and use its link as input for the embedding step.

In [19]:
# Embedding an image from a URL

image_url = 'https://gallica.bnf.fr/iiif/ark:/12148/bpt6k6069079/f2/775,369,1303,887/max/0/default.jpg'
external_embedding = client.tools.embed_image(image=image_url, target="image")
external_embedding


HTTPStatusError: Client error '403 Forbidden' for url 'https://gallica.bnf.fr/iiif/ark:/12148/bpt6k6069079/f2/775,369,1303,887/max/0/default.jpg'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

In [20]:
# Searching similar images from the embedded image URL

results = client.images.find(
  embedding=external_embedding,
  limit=5
)

results

NameError: name 'embedding' is not defined

# Conclusion

In this notebook, we explored how Impresso's models - **Open-CLIP for text-to-image search** and **DINOv2 for image-to-image similarity** - can be used to navigate historical visual collections.
Starting from simple and more descriptive queries, we saw how Open-CLIP retrieves radio programs and illustrated section titles across languages, before turning to DINOv2 to find visually similar images from a single reference example.
Together, these approaches show **how multimodal and visual embeddings can help us move beyond keyword search**.

---
## Project and License info

### Notebook credits [CreditLogo.png](https://credit.niso.org/)

**Writing - Original draft:**  Roman Kalyakin. **Conceptualization:** Marten Düring. **Software:** Roman Kalyakin. **Writing - Review & Editing**: Pauline Conti, Cao Vy. **Validation:** Marten Düring, Caio Mello. **Datalab editorial board:** Caio Mello (Managing), Cao Vy, Pauline Conti, Emanuela Boros, Marten Düring, Juri Opitz, Martin Grandjean, Estelle Bunout. **Data curation & Formal analysis:** Maud Ehrmann, Emanuela Boros, Pauline Conti, Simon Clematide, Juri Opitz, Andrianos Michail. **Methodology:** Roman Kalyakin. **Supervision:** Marten Düring. **Funding aquisition:** Maud Ehrmann, Simon Clematide, Marten Düring, Raphaëlle Ruppen Coutaz.

<br><a target="_blank" href="https://creativecommons.org/licenses/by/4.0/">
  <img src="https://mirrors.creativecommons.org/presskit/buttons/88x31/png/by.png"  width="100" alt="Open In Colab"/>
</a>

This notebook is published under [CC BY 4.0 License](https://creativecommons.org/licenses/by/4.0/)

For feedback on this notebook, please send an email to info@impresso-project.ch

### Impresso project

[Impresso - Media Monitoring of the Past](https://impresso-project.ch) is an interdisciplinary research project that aims to develop and consolidate tools for processing and exploring large collections of media archives across modalities, time, languages and national borders. The first project (2017-2021) was funded by the Swiss National Science Foundation under grant No. [CRSII5_173719](http://p3.snf.ch/project-173719) and the second project (2023-2027) by the SNSF under grant No. [CRSII5_213585](https://data.snf.ch/grants/grant/213585) and the Luxembourg National Research Fund under grant No. 17498891.
<br></br>
### License

All Impresso code is published open source under the [GNU Affero General Public License](https://github.com/impresso/impresso-pyindexation/blob/master/LICENSE) v3 or later.


---

<p align="center">
  <img src="https://github.com/impresso/impresso.github.io/blob/master/assets/images/3x1--Yellow-Impresso-Black-on-White--transparent.png?raw=true" width="350" alt="Impresso Project Logo"/>
</p>
